### 1.지도학습
 - 근거: Anomaly, 또는 Changepoint를 정답 라벨로 활용 가능
 - 정의: 입력 특성(X)과 정답(Y)를 모델에 제공하는 학습 방법 

### 2.비지도학습
 - 근거: 정상과 이상 데이터가 섞인 데이터셋 활용 가능  
 - 정의: 정상과 이상 데이터 혼합 데이터셋을 입력 특성(X)으로 모델에 제공하는 학습 방법

### 3.준지도학습
 - 근거: anomaly-free (무 이상 데이터) 데이터셋 활용 가능
 - 정의: 정상과 이상 중 한 가지 종류의 데이터 집합만 입력 특성(X)으로 모델에 제공하는 학습 방법

### 학습 방식 선택
#### - 준지도학습

### 실험 방향  
 - 목표1. 기존 벤치마크 성능을 재현해본다.  
 - 목표2. 재현 과정에서 겪은 경험을 토대로 최신 모델을 구현하여 성능을 측정해본다. 

#### 벤치마크 성능 재현 실험#1 (Isolation Forest) 

In [2]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
)

RANDOM_STATE = 42

# 노트북 위치에 따라 경로를 수정하세요.
# 예: SKAB 저장소 루트에서 실행한다면 Path("data")
DATA_ROOT = Path("assets/SKAB_0.9.0")

# "normal_only" 또는 "official_split"
EXPERIMENT_MODE = "official_split" # "normal_only"

# normal_only 모드에서 정상 Validation 점수의 상위 5%를 이상으로 판정
# 즉, Validation 정상 데이터 기준 목표 FAR 약 5%
TARGET_VALIDATION_FAR = 0.05

# Isolation Forest 설정
N_ESTIMATORS = 100
MAX_SAMPLES = "auto"
CONTAMINATION = "auto"

# StandardScaler는 Train 데이터에만 fit하고 Validation/Test에는 transform만 적용합니다.
USE_STANDARD_SCALER = False

print("DATA_ROOT:", DATA_ROOT.resolve())
print("EXPERIMENT_MODE:", EXPERIMENT_MODE)

DATA_ROOT: C:\github\CBNU.Master.1yr.apprentice\prj3\assets\SKAB_0.9.0
EXPERIMENT_MODE: official_split


In [26]:
LABEL_COLUMNS = {"anomaly", "changepoint"}
TIME_CANDIDATES = ("datetime", "timestamp", "time", "date")


def find_anomaly_free_file(data_root: Path) -> Path:
    candidates = list(data_root.rglob("anomaly-free.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"'anomaly-free.csv'를 찾지 못했습니다. DATA_ROOT를 확인하세요: {data_root.resolve()}"
        )
    if len(candidates) > 1:
        print("anomaly-free.csv 후보가 여러 개입니다. 첫 번째 파일을 사용합니다:")
        for p in candidates:
            print(" -", p)
    return candidates[0]


def read_skab_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=";")

    # 시간 컬럼을 인덱스로 설정 (있는 경우)
    time_col = next((c for c in TIME_CANDIDATES if c in df.columns), None)
    if time_col is not None:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.set_index(time_col)

    # 숫자형 컬럼 변환
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


anomaly_free_path = find_anomaly_free_file(DATA_ROOT)
anomaly_free_df = read_skab_csv(anomaly_free_path)

print("Anomaly-free file:", anomaly_free_path)
print("Shape:", anomaly_free_df.shape)
display(anomaly_free_df.head())
print("Columns:", anomaly_free_df.columns.tolist())

Anomaly-free file: assets\SKAB_0.9.0\anomaly-free\anomaly-free.csv
Shape: (9405, 8)


,Accelerometer1RMS,Accelerometer2RMS,Current,Pressure,Temperature,Thermocouple,Voltage,Volume Flow RateRMS
datetime,,,,,,,,
2020-02-08 13:30:47,0.202394,0.275154,2.16975,0.382638,90.6454,26.8508,238.852,122.664
2020-02-08 13:30:48,0.203153,0.277857,2.07999,-0.273216,90.7978,26.8639,227.943,122.338
2020-02-08 13:30:50,0.202054,0.275790,2.52577,0.382638,90.7730,26.8603,223.486,121.338
2020-02-08 13:30:51,0.203595,0.278101,2.49742,0.054711,90.8424,26.8616,244.904,121.664
2020-02-08 13:30:52,0.201889,0.276363,2.29194,0.710565,90.6664,26.8603,239.196,122.000


Columns: ['Accelerometer1RMS', 'Accelerometer2RMS', 'Current', 'Pressure', 'Temperature', 'Thermocouple', 'Voltage', 'Volume Flow RateRMS']


In [27]:
def get_feature_columns(df: pd.DataFrame):
    return [
        c for c in df.columns
        if c not in LABEL_COLUMNS and pd.api.types.is_numeric_dtype(df[c])
    ]


FEATURE_COLUMNS = get_feature_columns(anomaly_free_df)
if not FEATURE_COLUMNS:
    raise ValueError("센서 Feature 컬럼을 찾지 못했습니다. CSV 컬럼과 구분자를 확인하세요.")

print("Feature columns:", FEATURE_COLUMNS)
print("Anomaly-free missing values by feature:")
display(anomaly_free_df[FEATURE_COLUMNS].isna().sum().to_frame("missing_count"))

Feature columns: ['Accelerometer1RMS', 'Accelerometer2RMS', 'Current', 'Pressure', 'Temperature', 'Thermocouple', 'Voltage', 'Volume Flow RateRMS']
Anomaly-free missing values by feature:


,missing_count
Accelerometer1RMS,0
Accelerometer2RMS,0
Current,0
Pressure,0
Temperature,0
Thermocouple,0
Voltage,0
Volume Flow RateRMS,0


##### 전처리

In [30]:
from sklearn.impute import SimpleImputer


def chronological_split(df: pd.DataFrame, train_ratio=0.70, val_ratio=0.15):
    n = len(df)
    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    if train_end == 0 or val_end <= train_end or val_end >= n:
        raise ValueError(f"데이터 행 수가 분할에 충분하지 않습니다: {n}")

    return (
        df.iloc[:train_end].copy(),
        df.iloc[train_end:val_end].copy(),
        df.iloc[val_end:].copy(),
    )


def make_xy(df: pd.DataFrame, feature_columns):
    X = df.reindex(columns=feature_columns).copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


def fit_preprocessor(X_train: pd.DataFrame):
    imputer = SimpleImputer(strategy="median")
    X_train_imp = imputer.fit_transform(X_train)

    scaler = None
    if USE_STANDARD_SCALER:
        scaler = StandardScaler()
        X_train_ready = scaler.fit_transform(X_train_imp)
    else:
        X_train_ready = X_train_imp

    return imputer, scaler, X_train_ready


def transform_data(X: pd.DataFrame, imputer, scaler):
    X_imp = imputer.transform(X)
    if scaler is not None:
        return scaler.transform(X_imp)
    return X_imp


def build_iforest():

    # ! 벤치마크 리더보드 (F1:0.29) 재현을 위한 하이퍼파라미터
    RANDOM_STATE = 0
    CONTAMINATION = 0.0005
    
    return IsolationForest(
        n_estimators=N_ESTIMATORS,
        max_samples=MAX_SAMPLES,
        contamination=CONTAMINATION,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )


def anomaly_scores(model, X):
    # sklearn의 score_samples는 값이 작을수록 이상일 가능성이 큽니다.
    # 부호를 반전해 값이 클수록 이상인 anomaly score로 통일합니다.
    return -model.score_samples(X)


def binary_metrics(y_true, y_pred, y_score=None):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()

    result = {
        "n_samples": int(len(y_true)),
        "n_anomaly": int(np.sum(y_true == 1)),
        "n_normal": int(np.sum(y_true == 0)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "far": fp / (fp + tn) if (fp + tn) > 0 else np.nan,
        "mar": fn / (fn + tp) if (fn + tp) > 0 else np.nan,
    }
    if y_score is not None and len(np.unique(y_true)) == 2:
        result["average_precision"] = average_precision_score(y_true, y_score)
    else:
        result["average_precision"] = np.nan
    return result

##### 정상 데이터 학습 실험

In [13]:
def load_abnormal_experiment_files(data_root: Path, feature_columns):
    abnormal_frames = []
    allowed_dirs = {"valve1", "valve2", "other"}

    for folder in sorted(data_root.iterdir()):
        if folder.is_dir() and folder.name.lower() in allowed_dirs:
            for path in sorted(folder.rglob("*.csv")):
                df = read_skab_csv(path)

                if "anomaly" not in df.columns:
                    warnings.warn(f"anomaly 라벨이 없어 제외합니다: {path}")
                    continue

                # 센서 Feature와 anomaly 라벨만 사용
                missing_features = [c for c in feature_columns if c not in df.columns]
                if missing_features:
                    warnings.warn(
                        f"필수 Feature 누락으로 제외: {path} / {missing_features}"
                    )
                    continue

                frame = df[feature_columns + ["anomaly"]].copy()
                frame["anomaly"] = pd.to_numeric(frame["anomaly"], errors="coerce")
                frame = frame.dropna(subset=["anomaly"])
                frame["source_file"] = str(path.relative_to(data_root))
                abnormal_frames.append(frame)

    if not abnormal_frames:
        raise FileNotFoundError(
            "valve1/valve2/other에서 anomaly 라벨이 있는 CSV를 찾지 못했습니다."
        )

    return pd.concat(abnormal_frames, axis=0)


def run_normal_only_experiment():
    # anomaly-free는 정상 데이터로 취급합니다.
    normal_df = anomaly_free_df.copy()

    train_df, val_df, normal_test_df = chronological_split(
        normal_df, train_ratio=0.70, val_ratio=0.15
    )

    X_train = make_xy(train_df, FEATURE_COLUMNS)
    X_val = make_xy(val_df, FEATURE_COLUMNS)
    X_normal_test = make_xy(normal_test_df, FEATURE_COLUMNS)

    imputer, scaler, X_train_ready = fit_preprocessor(X_train)
    X_val_ready = transform_data(X_val, imputer, scaler)
    X_normal_test_ready = transform_data(X_normal_test, imputer, scaler)

    model = build_iforest()
    model.fit(X_train_ready)  # 라벨 없이 정상 Train 데이터만 학습

    val_scores = anomaly_scores(model, X_val_ready)

    # 높은 점수일수록 이상. 정상 Validation 점수의 상위 q 비율을 오탐 허용 영역으로 둠.
    threshold = float(np.quantile(val_scores, 1.0 - TARGET_VALIDATION_FAR))

    # 정상 Test
    normal_test_scores = anomaly_scores(model, X_normal_test_ready)
    normal_test_result = pd.DataFrame({
        "anomaly_score": normal_test_scores,
        "y_true": np.zeros(len(normal_test_scores), dtype=int),
        "source_file": "anomaly-free:test",
    }, index=normal_test_df.index)

    # 이상 실험 파일 Test
    abnormal_df = load_abnormal_experiment_files(DATA_ROOT, FEATURE_COLUMNS)
    X_abnormal = make_xy(abnormal_df, FEATURE_COLUMNS)
    X_abnormal_ready = transform_data(X_abnormal, imputer, scaler)
    abnormal_scores = anomaly_scores(model, X_abnormal_ready)

    abnormal_test_result = pd.DataFrame({
        "anomaly_score": abnormal_scores,
        "y_true": abnormal_df["anomaly"].astype(int).to_numpy(),
        "source_file": abnormal_df["source_file"].to_numpy(),
    }, index=abnormal_df.index)

    test_result = pd.concat(
        [normal_test_result, abnormal_test_result], axis=0
    )
    test_result["y_pred"] = (
        test_result["anomaly_score"] > threshold
    ).astype(int)

    metrics = binary_metrics(
        test_result["y_true"].to_numpy(),
        test_result["y_pred"].to_numpy(),
        test_result["anomaly_score"].to_numpy(),
    )

    print(f"Train rows: {len(train_df):,}")
    print(f"Validation rows: {len(val_df):,}")
    print(f"Normal Test rows: {len(normal_test_df):,}")
    print(f"Abnormal experiment Test rows: {len(abnormal_df):,}")
    print(f"Threshold (validation FAR target={TARGET_VALIDATION_FAR:.1%}): {threshold:.6f}")

    return model, imputer, scaler, threshold, test_result, metrics


if EXPERIMENT_MODE == "normal_only":
    model, imputer, scaler, threshold, test_result, metrics = run_normal_only_experiment()
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
    print("\nClassification report (point-wise anomaly label):")
    print(classification_report(
        test_result["y_true"],
        test_result["y_pred"],
        target_names=["normal", "anomaly"],
        zero_division=0,
    ))
    print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(test_result["y_true"], test_result["y_pred"], labels=[0, 1]))
else:
    print("normal_only 실행을 건너뜁니다. 아래 공식 분할 셀을 사용하세요.")

normal_only 실행을 건너뜁니다. 아래 공식 분할 셀을 사용하세요.


##### 공식 벤치마크 분할 방식 실험 (재현)

In [31]:
def run_official_split_experiment():
    all_files = []
    for folder_name in ("valve1", "valve2", "other"):
        folder = DATA_ROOT / folder_name
        if folder.exists():
            all_files.extend(sorted(folder.rglob("*.csv")))

    if not all_files:
        raise FileNotFoundError("valve1/valve2/other 데이터 파일을 찾지 못했습니다.")

    rows = []
    per_file_results = []

    for path in all_files:
        df = read_skab_csv(path)

        if not set(FEATURE_COLUMNS).issubset(df.columns):
            warnings.warn(f"Feature 누락으로 제외: {path}")
            continue
        if "anomaly" not in df.columns:
            warnings.warn(f"anomaly 라벨 누락으로 제외: {path}")
            continue

        if len(df) <= 400:
            warnings.warn(f"400행 이하라 제외: {path} ({len(df)} rows)")
            continue

        X = make_xy(df, FEATURE_COLUMNS)
        y = pd.to_numeric(df["anomaly"], errors="coerce")

        X_train = X.iloc[:400]
        X_test = X.iloc[400:]
        y_test = y.iloc[400:].fillna(0).astype(int)

        # 공식 분할은 파일마다 독립적으로 학습합니다.
        imputer, scaler, X_train_ready = fit_preprocessor(X_train)

        # 공식 코드와 동일한 raw feature를 사용하려면 USE_STANDARD_SCALER=False로 설정하세요.
        X_test_ready = transform_data(X_test, imputer, scaler)

        model = build_iforest()
        model.fit(X_train_ready)  # y_train 미사용

        prediction = pd.Series(
            model.predict(X_test_ready) * -1,
            index=X_test.index,
        )

        # ! 변경 코드
        prediction = (
            prediction
            .rolling(3)
            .median()
            .fillna(0)
            .replace(-1, 0)
            .astype(int)
        )

        y_pred = prediction.to_numpy()

        # ! 기존 코드
        # y_pred = (model.predict(X_test_ready) == -1).astype(int)
        scores = anomaly_scores(model, X_test_ready)

        file_result = binary_metrics(y_test.to_numpy(), y_pred, scores)
        file_result["file"] = str(path.relative_to(DATA_ROOT))
        per_file_results.append(file_result)

        rows.append(pd.DataFrame({
            "y_true": y_test.to_numpy(),
            "y_pred": y_pred,
            "anomaly_score": scores,
            "source_file": str(path.relative_to(DATA_ROOT)),
        }, index=X_test.index))

    if not rows:
        raise RuntimeError("평가 가능한 데이터 파일이 없습니다.")

    result = pd.concat(rows, axis=0)
    pooled_metrics = binary_metrics(
        result["y_true"].to_numpy(),
        result["y_pred"].to_numpy(),
        result["anomaly_score"].to_numpy(),
    )

    return result, pd.DataFrame(per_file_results), pooled_metrics


if EXPERIMENT_MODE == "official_split":
    # 공식 분할 재현 시 스케일링 여부를 공식 노트북 구현과 맞추세요.
    official_result, per_file_metrics, official_metrics = run_official_split_experiment()
    print("Pooled metrics:")
    display(pd.DataFrame([official_metrics]).T.rename(columns={0: "value"}))
    print("Per-file metrics:")
    display(per_file_metrics)
    print("Pooled confusion matrix [ [TN, FP], [FN, TP] ]:")
    print(confusion_matrix(
        official_result["y_true"],
        official_result["y_pred"],
        labels=[0, 1],
    ))

Pooled metrics:


,value
n_samples,23801.000000
n_anomaly,12771.000000
n_normal,11030.000000
accuracy,0.543381
precision,0.885691
recall,0.171091
f1,0.286783
far,0.025567
mar,0.828909
average_precision,0.746174


Per-file metrics:


,n_samples,n_anomaly,n_normal,accuracy,precision,recall,f1,far,mar,average_precision,file
0,747,401,346,0.467202,0.800000,0.009975,0.019704,0.002890,0.990025,0.592982,valve1\0.csv
1,745,402,343,0.492617,0.928571,0.064677,0.120930,0.005831,0.935323,0.759137,valve1\1.csv
2,746,401,345,0.591153,1.000000,0.239401,0.386318,0.000000,0.760599,0.889680,valve1\10.csv
3,741,399,342,0.462888,0.666667,0.005013,0.009950,0.002924,0.994987,0.650505,valve1\11.csv
4,740,399,341,0.821622,0.940594,0.714286,0.811966,0.052786,0.285714,0.897851,valve1\12.csv
5,740,399,341,0.629730,1.000000,0.313283,0.477099,0.000000,0.686717,0.909686,valve1\13.csv
6,739,399,340,0.589986,1.000000,0.240602,0.387879,0.000000,0.759398,0.853648,valve1\14.csv
7,750,404,346,0.461333,0.000000,0.000000,0.000000,0.000000,1.000000,0.684940,valve1\15.csv
8,675,337,338,0.471111,0.142857,0.011869,0.021918,0.071006,0.988131,0.524747,valve1\2.csv
9,748,404,344,0.465241,1.000000,0.009901,0.019608,0.000000,0.990099,0.788731,valve1\3.csv


Pooled confusion matrix [ [TN, FP], [FN, TP] ]:
[[10748   282]
 [10586  2185]]


##### 결과 시각화 (정상 데이터셋에 한해)

In [3]:
if EXPERIMENT_MODE == "normal_only":
    plt.figure(figsize=(12, 5))
    plt.hist(
        test_result.loc[test_result["y_true"] == 0, "anomaly_score"],
        bins=60, alpha=0.6, label="Normal (test)", density=True
    )
    plt.hist(
        test_result.loc[test_result["y_true"] == 1, "anomaly_score"],
        bins=60, alpha=0.6, label="Anomaly (test)", density=True
    )
    plt.axvline(
        threshold, linestyle="--", linewidth=2,
        label=f"Threshold = {threshold:.4f}"
    )
    plt.xlabel("Anomaly score (-score_samples)")
    plt.ylabel("Density")
    plt.title("Isolation Forest — Test anomaly score distribution")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    # 시간 순서에 따른 점수 일부 확인
    plot_n = min(2000, len(test_result))
    plot_df = test_result.iloc[:plot_n]

    plt.figure(figsize=(14, 4))
    plt.plot(np.arange(len(plot_df)), plot_df["anomaly_score"], linewidth=0.8)
    plt.axhline(threshold, color="red", linestyle="--", label="Threshold")
    plt.scatter(
        np.where(plot_df["y_true"].to_numpy() == 1)[0],
        plot_df.loc[plot_df["y_true"] == 1, "anomaly_score"],
        s=8, label="Ground-truth anomaly"
    )
    plt.xlabel("Test sample index")
    plt.ylabel("Anomaly score")
    plt.title("Anomaly score over test samples (first segment)")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

#### 벤치마크 성능 재현 실험#2 (LSTM-AE) 

In [ ]:
# 필요 시 설치
# %pip install numpy pandas scikit-learn matplotlib torch

from pathlib import Path
import copy
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

# -----------------------------
# 실험 설정
# -----------------------------
SEED = 0
DATA_ROOT = Path("assets/SKAB_0.9.0")
EXPERIMENT_MODE = "official_split"   # "normal_only" 또는 "official_split"

WINDOW_SIZE = 30
HIDDEN_SIZE = 64
LATENT_SIZE = 16
NUM_LAYERS = 1
DROPOUT = 0.0

EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
PATIENCE = 7

# normal_only: 정상 Validation 점수의 상위 5%를 이상으로 분류
TARGET_VALIDATION_FAR = 0.05

# official_split: train reconstruction error의 상위 95%를 threshold로 사용
OFFICIAL_THRESHOLD_QUANTILE = 0.95

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("DATA_ROOT:", DATA_ROOT.resolve())
print("MODE:", EXPERIMENT_MODE)
print("DEVICE:", DEVICE)
print("PyTorch:", torch.__version__)


DATA_ROOT: C:\github\CBNU.Master.1yr.apprentice\prj3\assets\SKAB_0.9.0
MODE: official_split
DEVICE: cuda
PyTorch: 2.15.0+cu132


In [11]:
LABEL_COLUMNS = {"anomaly", "changepoint"}
TIME_CANDIDATES = ("datetime", "timestamp", "time", "date")

def find_anomaly_free_file(data_root: Path) -> Path:
    candidates = list(data_root.rglob("anomaly-free.csv"))
    if not candidates:
        raise FileNotFoundError(
            f"'anomaly-free.csv'를 찾지 못했습니다. DATA_ROOT={data_root.resolve()}"
        )
    return candidates[0]

def read_skab_csv(path: Path) -> pd.DataFrame:
    # SKAB CSV는 세미콜론 구분자를 사용
    df = pd.read_csv(path, sep=";")
    time_col = next((c for c in TIME_CANDIDATES if c in df.columns), None)
    if time_col is not None:
        df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.set_index(time_col)
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

def get_feature_columns(df: pd.DataFrame):
    cols = [
        c for c in df.columns
        if c.lower() not in LABEL_COLUMNS and pd.api.types.is_numeric_dtype(df[c])
    ]
    if not cols:
        raise ValueError("수치형 센서 Feature를 찾지 못했습니다.")
    return cols

def make_xy(df: pd.DataFrame, feature_columns):
    return df.loc[:, feature_columns].replace([np.inf, -np.inf], np.nan)

def fit_preprocessor(X_train):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    X_imp = imputer.fit_transform(X_train)
    X_scaled = scaler.fit_transform(X_imp)
    return imputer, scaler, X_scaled

def transform_data(X, imputer, scaler):
    return scaler.transform(imputer.transform(X))

anomaly_free_path = find_anomaly_free_file(DATA_ROOT)
normal_df = read_skab_csv(anomaly_free_path)
FEATURE_COLUMNS = get_feature_columns(normal_df)

print("Anomaly-free:", anomaly_free_path)
print("Rows:", len(normal_df), "| Features:", FEATURE_COLUMNS)


Anomaly-free: assets\SKAB_0.9.0\anomaly-free\anomaly-free.csv
Rows: 9405 | Features: ['Accelerometer1RMS', 'Accelerometer2RMS', 'Current', 'Pressure', 'Temperature', 'Thermocouple', 'Voltage', 'Volume Flow RateRMS']


##### 시퀀스 생성

In [12]:
def make_windows(X, window_size=WINDOW_SIZE, target_start=None):
    X = np.asarray(X, dtype=np.float32)
    if target_start is None:
        target_start = window_size - 1

    windows, target_indices = [], []
    for end_idx in range(max(window_size - 1, target_start), len(X)):
        start_idx = end_idx - window_size + 1
        if start_idx < 0:
            continue
        windows.append(X[start_idx:end_idx + 1])
        target_indices.append(end_idx)

    if not windows:
        return (
            np.empty((0, window_size, X.shape[1]), dtype=np.float32),
            np.array([], dtype=int)
        )
    return np.stack(windows), np.asarray(target_indices, dtype=int)

def windows_for_range(X, start_idx, end_idx, window_size=WINDOW_SIZE):
    # [start_idx, end_idx) 구간의 각 target을 평가하되,
    # window context로 이전 시점 데이터도 허용
    X = np.asarray(X, dtype=np.float32)
    windows, target_indices = [], []
    for target_idx in range(start_idx, end_idx):
        left = target_idx - window_size + 1
        if left < 0:
            continue
        windows.append(X[left:target_idx + 1])
        target_indices.append(target_idx)
    if not windows:
        return np.empty((0, window_size, X.shape[1]), np.float32), np.array([], int)
    return np.stack(windows), np.asarray(target_indices, dtype=int)

def score_to_frame(df, scores, target_indices, y=None, source=""):
    out = pd.DataFrame({
        "anomaly_score": scores,
        "source_file": source,
    }, index=df.index[target_indices])
    if y is not None:
        out["y_true"] = np.asarray(y)[target_indices].astype(int)
    return out


##### LSTM Autoencoder 모델 및 학습

In [13]:
class LSTMAutoencoder(nn.Module):
    def __init__(
        self, n_features, hidden_size=HIDDEN_SIZE,
        latent_size=LATENT_SIZE, num_layers=NUM_LAYERS, dropout=DROPOUT
    ):
        super().__init__()
        recurrent_dropout = dropout if num_layers > 1 else 0.0

        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )
        self.to_latent = nn.Linear(hidden_size, latent_size)
        self.from_latent = nn.Linear(latent_size, hidden_size)
        self.decoder = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=recurrent_dropout,
        )
        self.output_layer = nn.Linear(hidden_size, n_features)

    def forward(self, x):
        _, (h_n, _) = self.encoder(x)
        latent = self.to_latent(h_n[-1])              # (batch, latent)
        repeated = self.from_latent(latent).unsqueeze(1)
        repeated = repeated.repeat(1, x.size(1), 1)  # (batch, seq, hidden)
        decoded, _ = self.decoder(repeated)
        return self.output_layer(decoded)             # (batch, seq, features)

def train_lstm_ae(X_train_seq, X_val_seq, n_features):
    if len(X_train_seq) == 0 or len(X_val_seq) == 0:
        raise ValueError("학습/검증 시퀀스가 부족합니다. WINDOW_SIZE 또는 분할 비율을 확인하세요.")

    model = LSTMAutoencoder(n_features).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.MSELoss()

    train_ds = TensorDataset(torch.tensor(X_train_seq, dtype=torch.float32))
    val_ds = TensorDataset(torch.tensor(X_val_seq, dtype=torch.float32))
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    best_state = copy.deepcopy(model.state_dict())
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    history = {"loss": [], "val_loss": []}

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for (batch,) in train_loader:
            batch = batch.to(DEVICE)
            optimizer.zero_grad()
            reconstructed = model(batch)
            loss = criterion(reconstructed, batch)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for (batch,) in val_loader:
                batch = batch.to(DEVICE)
                reconstructed = model(batch)
                val_losses.append(criterion(reconstructed, batch).item())

        train_loss = float(np.mean(train_losses))
        val_loss = float(np.mean(val_losses))
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:03d} | loss={train_loss:.6f} | val_loss={val_loss:.6f}")

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    model.load_state_dict(best_state)
    return model, history

def reconstruction_scores(model, X_seq):
    if len(X_seq) == 0:
        return np.array([], dtype=float)

    model.eval()
    loader = DataLoader(
        TensorDataset(torch.tensor(X_seq, dtype=torch.float32)),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    scores = []
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            reconstructed = model(batch)
            # 마지막 timestep의 feature별 MSE
            point_error = torch.mean((reconstructed[:, -1, :] - batch[:, -1, :]) ** 2, dim=1)
            scores.extend(point_error.cpu().numpy())
    return np.asarray(scores, dtype=float)


##### 평가 지표

In [14]:
def binary_metrics(y_true, y_pred, scores=None):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    result = {
        "n_samples": int(len(y_true)),
        "n_normal": int(np.sum(y_true == 0)),
        "n_anomaly": int(np.sum(y_true == 1)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "far": fp / (fp + tn) if (fp + tn) else np.nan,
        "mar": fn / (fn + tp) if (fn + tp) else np.nan,
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }
    if scores is not None and len(np.unique(y_true)) == 2:
        result["average_precision"] = average_precision_score(y_true, scores)
    return result

def list_abnormal_files(data_root):
    files = []
    for folder_name in ("valve1", "valve2", "other"):
        folder = data_root / folder_name
        if folder.exists():
            files.extend(sorted(folder.rglob("*.csv")))
    return files


##### Normal Only 실험

In [ ]:
def run_normal_only():
    df = normal_df.copy()
    X_df = make_xy(df, FEATURE_COLUMNS)

    n = len(X_df)
    train_end = int(n * 0.70)
    val_end = int(n * 0.85)

    X_train_df = X_df.iloc[:train_end]
    imputer, scaler, X_train = fit_preprocessor(X_train_df)
    X_all = transform_data(X_df, imputer, scaler)

    # Train windows
    X_train_seq, _ = windows_for_range(X_all, WINDOW_SIZE - 1, train_end)
    # Validation targets use preceding train context
    X_val_seq, val_idx = windows_for_range(X_all, train_end, val_end)

    model, history = train_lstm_ae(X_train_seq, X_val_seq, X_all.shape[1])
    val_scores = reconstruction_scores(model, X_val_seq)
    threshold = float(np.quantile(val_scores, 1.0 - TARGET_VALIDATION_FAR))

    results = []

    # 정상 Test 구간
    X_normal_test_seq, normal_test_idx = windows_for_range(X_all, val_end, n)
    normal_scores = reconstruction_scores(model, X_normal_test_seq)
    normal_y = pd.to_numeric(df["anomaly"], errors="coerce").fillna(0).to_numpy()
    normal_result = score_to_frame(
        df, normal_scores, normal_test_idx, normal_y, "anomaly-free.csv"
    )
    normal_result["y_pred"] = (normal_result["anomaly_score"] >= threshold).astype(int)
    results.append(normal_result)

    # 이상 실험 파일별 평가
    for path in list_abnormal_files(DATA_ROOT):
        adf = read_skab_csv(path)
        if not set(FEATURE_COLUMNS).issubset(adf.columns) or "anomaly" not in adf.columns:
            warnings.warn(f"Feature/anomaly 라벨 누락으로 제외: {path}")
            continue

        X_adf = make_xy(adf, FEATURE_COLUMNS)
        X_adf_scaled = transform_data(X_adf, imputer, scaler)
        X_seq, target_idx = make_windows(X_adf_scaled)
        scores = reconstruction_scores(model, X_seq)
        y = pd.to_numeric(adf["anomaly"], errors="coerce").fillna(0).to_numpy()

        result = score_to_frame(
            adf, scores, target_idx, y, str(path.relative_to(DATA_ROOT))
        )
        result["y_pred"] = (result["anomaly_score"] >= threshold).astype(int)
        results.append(result)

    combined = pd.concat(results, axis=0)
    metrics = binary_metrics(
        combined["y_true"].to_numpy(),
        combined["y_pred"].to_numpy(),
        combined["anomaly_score"].to_numpy(),
    )
    return model, history, threshold, combined, metrics

if EXPERIMENT_MODE == "normal_only":
    model, history, threshold, test_result, metrics = run_normal_only()
    print("Threshold:", threshold)
    display(pd.DataFrame([metrics]).T.rename(columns={0: "value"}))
else:
    print("normal_only 실행을 건너뜁니다.")


##### Official split 실험

In [15]:
def run_official_split():
    rows = []
    per_file = []

    for path in list_abnormal_files(DATA_ROOT):
        df = read_skab_csv(path)
        if not set(FEATURE_COLUMNS).issubset(df.columns) or "anomaly" not in df.columns:
            warnings.warn(f"Feature/anomaly 라벨 누락으로 제외: {path}")
            continue
        if len(df) <= 400 + WINDOW_SIZE:
            warnings.warn(f"데이터 길이 부족으로 제외: {path}")
            continue

        X_df = make_xy(df, FEATURE_COLUMNS)
        y = pd.to_numeric(df["anomaly"], errors="coerce").fillna(0).to_numpy()

        X_train_df = X_df.iloc[:400]
        imputer, scaler, X_train_raw = fit_preprocessor(X_train_df)
        X_all = transform_data(X_df, imputer, scaler)

        # 400행 학습 구간 내 시간 순서로 train/validation window 분리
        train_seq_all, train_targets = windows_for_range(
            X_all, WINDOW_SIZE - 1, 400
        )
        split = max(1, int(len(train_seq_all) * 0.80))
        X_fit_seq = train_seq_all[:split]
        X_val_seq = train_seq_all[split:]
        if len(X_val_seq) == 0:
            X_val_seq = train_seq_all[-1:]
            X_fit_seq = train_seq_all[:-1]

        model, history = train_lstm_ae(X_fit_seq, X_val_seq, X_all.shape[1])

        # threshold는 학습 구간의 reconstruction score 분포로 설정
        train_scores = reconstruction_scores(model, train_seq_all)
        threshold = float(np.quantile(train_scores, OFFICIAL_THRESHOLD_QUANTILE))

        # Test 시작 시점부터 점수 계산. 앞 400행은 window context로만 사용
        X_test_seq, test_idx = windows_for_range(X_all, 400, len(df))
        scores = reconstruction_scores(model, X_test_seq)
        y_test = y[test_idx]
        y_pred = (scores >= threshold).astype(int)

        file_metrics = binary_metrics(y_test, y_pred, scores)
        file_metrics["file"] = str(path.relative_to(DATA_ROOT))
        file_metrics["threshold"] = threshold
        per_file.append(file_metrics)

        rows.append(pd.DataFrame({
            "y_true": y_test,
            "y_pred": y_pred,
            "anomaly_score": scores,
            "source_file": str(path.relative_to(DATA_ROOT)),
        }, index=df.index[test_idx]))

    if not rows:
        raise RuntimeError("평가 가능한 이상 실험 파일이 없습니다.")

    combined = pd.concat(rows)
    pooled = binary_metrics(
        combined["y_true"].to_numpy(),
        combined["y_pred"].to_numpy(),
        combined["anomaly_score"].to_numpy(),
    )
    return combined, pd.DataFrame(per_file), pooled

if EXPERIMENT_MODE == "official_split":
    official_result, per_file_metrics, official_metrics = run_official_split()
    print("Pooled metrics:")
    display(pd.DataFrame([official_metrics]).T.rename(columns={0: "value"}))
    print("Per-file metrics:")
    display(per_file_metrics)
else:
    print("official_split 실행을 건너뜁니다.")


Epoch 001 | loss=0.995385 | val_loss=1.106345
Epoch 005 | loss=0.913789 | val_loss=1.204308
Epoch 010 | loss=0.796702 | val_loss=0.786332
Epoch 015 | loss=0.777326 | val_loss=0.787626
Early stopping at epoch 18
Epoch 001 | loss=1.008966 | val_loss=1.030688
Epoch 005 | loss=0.940963 | val_loss=1.123216
Early stopping at epoch 8
Epoch 001 | loss=0.938676 | val_loss=1.202635
Epoch 005 | loss=0.905136 | val_loss=1.313965
Early stopping at epoch 8
Epoch 001 | loss=0.997769 | val_loss=0.873753
Epoch 005 | loss=0.912516 | val_loss=0.952143
Epoch 010 | loss=0.755528 | val_loss=0.888792
Epoch 015 | loss=0.735064 | val_loss=0.836723
Early stopping at epoch 15
Epoch 001 | loss=1.016537 | val_loss=0.950754
Epoch 005 | loss=0.976083 | val_loss=0.991855
Early stopping at epoch 8
Epoch 001 | loss=0.991914 | val_loss=0.983141
Epoch 005 | loss=0.950756 | val_loss=1.100471
Epoch 010 | loss=0.803421 | val_loss=0.889671
Epoch 015 | loss=0.755935 | val_loss=0.816552
Epoch 020 | loss=0.710977 | val_loss=0.8

,value
n_samples,23801.000000
n_normal,11030.000000
n_anomaly,12771.000000
accuracy,0.704214
precision,0.669767
recall,0.885209
f1,0.762563
far,0.505349
mar,0.114791
tn,5456.000000


Per-file metrics:


,n_samples,n_normal,n_anomaly,accuracy,precision,recall,f1,far,mar,tn,fp,fn,tp,average_precision,file,threshold
0,747,346,401,0.685408,0.644599,0.922693,0.758974,0.589595,0.077307,142,204,31,370,0.616082,valve1\0.csv,1.626186
1,745,343,402,0.642953,0.639344,0.776119,0.701124,0.513120,0.223881,167,176,90,312,0.600032,valve1\1.csv,2.135488
2,746,345,401,0.845845,0.829493,0.897756,0.862275,0.214493,0.102244,271,74,41,360,0.936123,valve1\10.csv,1.849440
3,741,342,399,0.771930,0.821229,0.736842,0.776750,0.187135,0.263158,278,64,105,294,0.892248,valve1\11.csv,1.874734
4,740,341,399,0.552703,0.546575,1.000000,0.706820,0.970674,0.000000,10,331,0,399,0.606240,valve1\12.csv,1.853948
5,740,341,399,0.817568,0.812796,0.859649,0.835566,0.231672,0.140351,262,79,56,343,0.952675,valve1\13.csv,1.422354
6,739,340,399,0.828146,0.838308,0.844612,0.841448,0.191176,0.155388,275,65,62,337,0.941227,valve1\14.csv,1.463717
7,750,346,404,0.732000,0.674099,0.972772,0.796353,0.549133,0.027228,156,190,11,393,0.968095,valve1\15.csv,1.551830
8,675,338,337,0.503704,0.501488,1.000000,0.667988,0.991124,0.000000,3,335,0,337,0.437351,valve1\2.csv,2.088757
9,748,344,404,0.735294,0.943966,0.542079,0.688679,0.037791,0.457921,331,13,185,219,0.892707,valve1\3.csv,2.310757


##### 학습 곡선 및 anomaly score

In [ ]:
if EXPERIMENT_MODE == "normal_only":
    plt.figure(figsize=(8, 4))
    plt.plot(history["loss"], label="Train loss")
    plt.plot(history["val_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Reconstruction MSE")
    plt.title("LSTM-AE training history")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()

    plt.figure(figsize=(10, 4))
    for label, group in test_result.groupby("y_true"):
        plt.hist(
            group["anomaly_score"], bins=60, alpha=0.5, density=True,
            label="Anomaly" if label == 1 else "Normal"
        )
    plt.axvline(threshold, linestyle="--", label=f"Threshold={threshold:.5f}")
    plt.xlabel("Reconstruction error (last timestep MSE)")
    plt.ylabel("Density")
    plt.title("LSTM-AE test score distribution")
    plt.legend()
    plt.grid(alpha=0.2)
    plt.show()


#### 벤치마크 성능 재현 실험#3 (Conv-AE)

#### 최신 모델 성능 실험#1 (USAD;2020)